In [ ]:
# ==============================================================================
# CELDA 0: RUTAS DEL REPOSITORIO
# Este cuaderno leia sus datos desde Google Drive. Ahora los lee del propio
# repositorio, de modo que corre en cualquier clon sin configuracion previa.
# La raiz se resuelve buscando hacia arriba: funciona igual si el cuaderno se
# ejecuta desde su carpeta o desde la raiz del proyecto.
# ==============================================================================
from pathlib import Path

def _raiz_del_repositorio() -> Path:
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / 'pyproject.toml').exists() and (carpeta / 'quanta').is_dir():
            return carpeta
    raise RuntimeError(
        'No se encontro la raiz del repositorio. Ejecute el cuaderno desde '
        'dentro del proyecto indice-sned.'
    )

RAIZ            = _raiz_del_repositorio()
RUTA_RAW        = (RAIZ / 'data' / 'raw').as_posix() + '/'
RUTA_PROCESADOS = (RAIZ / 'data' / 'processed').as_posix() + '/'
RUTA_REGISTRO   = (RAIZ / 'models' / 'registry').as_posix() + '/'
RUTA_METADATOS  = (RAIZ / 'models' / 'metadata').as_posix() + '/'

print('Raiz del repositorio:', RAIZ)


In [4]:
# ==============================================================================
# CELDA 1: LIBRERÍAS Y ENTORNO
# ==============================================================================
import pandas as pd
import numpy as np
import openpyxl


print("Entorno listo. Pandas:", pd.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Entorno listo. Pandas: 2.2.2


In [6]:
# ==============================================================================
# CELDA 2: INGESTA SEGURA DE ARCHIVOS IDPS (DATAFRAMES BRUTOS)
# ==============================================================================
RUTA_BASE = RUTA_RAW + 'idps/'

archivos_idps = {
    'idps_4b_2016': 'idps4b2016_rbd.xls',
    'idps_2m_2016': 'idps2m2016_rbd.xls',
    'idps_4b_2017': 'idps4b2017_rbd_final.xlsx',
    'idps_8b_2017': 'idps8b2017_rbd_final.xlsx',
    'idps_2m_2017': 'idps2m2017_rbd_final.xlsx',
    'idps_4b_2018': 'idps_4b2018.xlsx',
    'idps_6b_2018': 'idps_6b2018.xlsx',
    'idps_2m_2018': 'idps_2m2018.xlsx',
    'idps_8b_2019': 'idps19_rbd.xlsx',   # único nivel medido en 2019 (8ºB), por el estallido social
    'idps_4b_2022': 'idps4b2022_rbd_final.xlsx',
    'idps_2m_2022': 'idps2m2022_rbd_final.xlsx',
    'idps_4b_2023': 'idps4b2023_rbd_final.xlsx',
    'idps_2m_2023': 'idps2m2023_rbd_final.xlsx',
    'idps_4b_2024': 'idps4B2024_rbd_final.xlsx',
    'idps_6b_2024': 'idps6B2024_rbd_preliminar.xlsx',
    'idps_2m_2024': 'idps2M2024_rbd_final.xlsx',
    'idps_4b_2025': 'idps4B2025_rbd_preliminar.xlsx',
    'idps_8b_2025': 'idps8B2025_rbd_preliminar.xlsx',
    'idps_2m_2025': 'idps2M2025_rbd_preliminar.xlsx',
}

dfs_idps_brutos = {}

for clave, nombre_archivo in archivos_idps.items():
    ruta_completa = RUTA_BASE + nombre_archivo
    try:
        if nombre_archivo.lower().endswith('.csv'):
            dfs_idps_brutos[clave] = pd.read_csv(ruta_completa, encoding='utf-8-sig', sep=';', low_memory=False)
        elif nombre_archivo.lower().endswith('.xls'):
            dfs_idps_brutos[clave] = pd.read_excel(ruta_completa, engine='xlrd')
        else:
            dfs_idps_brutos[clave] = pd.read_excel(ruta_completa, engine='openpyxl')
        print(f"OK      | {clave:<16} | {dfs_idps_brutos[clave].shape[0]:>6} filas x {dfs_idps_brutos[clave].shape[1]:>3} columnas")
    except FileNotFoundError:
        print(f"FALTA   | {clave:<16} | No se encontró: {nombre_archivo}")
    except Exception as e:
        print(f"ERROR   | {clave:<16} | {type(e).__name__}: {e}")

print(f"\nTotal de DataFrames cargados en memoria: {len(dfs_idps_brutos)} de {len(archivos_idps)}")

print("\n--- Columnas por archivo (para diseñar la limpieza) ---")
for clave, d in dfs_idps_brutos.items():
    print(f"\n{clave}: {d.columns.tolist()}")

OK      | idps_4b_2016     |   7390 filas x   7 columnas
OK      | idps_2m_2016     |   2891 filas x   7 columnas
OK      | idps_4b_2017     |   7384 filas x  18 columnas
OK      | idps_8b_2017     |   5982 filas x  18 columnas
OK      | idps_2m_2017     |   2912 filas x  18 columnas
OK      | idps_4b_2018     |   7414 filas x  29 columnas
OK      | idps_6b_2018     |   7322 filas x  29 columnas
OK      | idps_2m_2018     |   2935 filas x  29 columnas
OK      | idps_8b_2019     |   5960 filas x  18 columnas
OK      | idps_4b_2022     |  28660 filas x  16 columnas
OK      | idps_2m_2022     |  11904 filas x  16 columnas
OK      | idps_4b_2023     |  28172 filas x  18 columnas
OK      | idps_2m_2023     |  11888 filas x  18 columnas
OK      | idps_4b_2024     |  28740 filas x  22 columnas
OK      | idps_6b_2024     |  28344 filas x  22 columnas
OK      | idps_2m_2024     |  12000 filas x  22 columnas
OK      | idps_4b_2025     |  26868 filas x  24 columnas
OK      | idps_8b_2025     |  2

In [7]:
# ==============================================================================
# VERIFICACIÓN: VALORES REALES DE LA COLUMNA 'ind' / 'id_indicador' (formato largo)
# ==============================================================================
print("--- Valores únicos de 'ind' en 2022 (formato largo antiguo) ---")
print(dfs_idps_brutos['idps_4b_2022']['ind'].value_counts())

print("\n--- Valores únicos de 'ind' en 2024 (formato largo, con dif/sigdif) ---")
print(dfs_idps_brutos['idps_4b_2024']['ind'].value_counts())

print("\n--- Valores únicos de 'id_indicador' en 2025 (formato largo más reciente) ---")
print(dfs_idps_brutos['idps_4b_2025']['id_indicador'].value_counts())

# Muestra de una fila completa para entender qué son dif/sigdif/difgru/sigdifgru
print("\n--- Muestra completa: un solo colegio en 2024 (todas sus filas) ---")
rbd_ejemplo = dfs_idps_brutos['idps_4b_2024']['rbd'].iloc[0]
print(dfs_idps_brutos['idps_4b_2024'][dfs_idps_brutos['idps_4b_2024']['rbd'] == rbd_ejemplo].to_string(index=False))

--- Valores únicos de 'ind' en 2022 (formato largo antiguo) ---
ind
AM    7165
CC    7165
HV    7165
PF    7165
Name: count, dtype: int64

--- Valores únicos de 'ind' en 2024 (formato largo, con dif/sigdif) ---
ind
AM    7185
CC    7185
HV    7185
PF    7185
Name: count, dtype: int64

--- Valores únicos de 'id_indicador' en 2025 (formato largo más reciente) ---
id_indicador
1    6717
2    6717
3    6717
4    6717
Name: count, dtype: int64

--- Muestra completa: un solo colegio en 2024 (todas sus filas) ---
 agno  rbd ind  prom  dif  sigdif  difgru  sigdifgru                  nom_rbd  cod_reg_rbd           nom_reg_rbd  cod_pro_rbd nom_pro_rbd  cod_com_rbd nom_com_rbd nom_deprov_rbd  cod_depe2  cod_grupo  cod_rural_rbd      codigo_bdd  fecha_bbdd grado
 2024    5  AM  75.0 -1.0     0.0     1.0        0.0 JOVINA NARANJO FERNANDEZ           15 DE ARICA Y PARINACOTA          151       ARICA        15101       ARICA          Arica          4        3.0              1 final20250625v1    20250

In [9]:
# ==============================================================================
# CELDA 3: LIMPIEZA, PIVOT (LARGO->ANCHO) Y ESTANDARIZACIÓN IDPS
# ==============================================================================
MAPEO_ID_INDICADOR = {1: 'AM', 2: 'CC', 3: 'PF', 4: 'HV'}

dfs_idps_limpios = {}

for clave, df_bruto in dfs_idps_brutos.items():
    d = df_bruto.copy()
    nivel = clave.split('_')[1]

    # --- Normalización de nombres de columna a minúscula (2018 viene en MAYÚSCULA) ---
    d.columns = [c.lower() for c in d.columns]

    # --- Caso A: formato LARGO ---
    if 'ind' in d.columns or 'id_indicador' in d.columns:
        col_indicador = 'ind' if 'ind' in d.columns else 'id_indicador'

        if pd.api.types.is_numeric_dtype(d[col_indicador]):
            d[col_indicador] = d[col_indicador].map(MAPEO_ID_INDICADOR)
        else:
            d[col_indicador] = d[col_indicador].astype(str).str.upper().str.strip()

        cols_meta = [c for c in ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'cod_rural_rbd'] if c in d.columns]
        meta = d.groupby('rbd', as_index=False)[cols_meta[1:]].first() if len(cols_meta) > 1 else d[['rbd']].drop_duplicates()

        pivot = d.pivot_table(index='rbd', columns=col_indicador, values='prom', aggfunc='first').reset_index()
        pivot.columns.name = None

        df = pd.merge(meta, pivot, on='rbd', how='right')
        df = df.rename(columns={sigla: f'idps_{sigla.lower()}_{nivel}' for sigla in ['AM', 'CC', 'PF', 'HV']})

    # --- Caso B: formato ANCHO ya consolidado (2016-2019) ---
    else:
        df = d.copy()
        cols_meta = [c for c in ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'cod_rural_rbd'] if c in df.columns]

        renombres = {}
        for sigla in ['am', 'cc', 'pf', 'hv']:
            for patron in [f'ind_{sigla}', f'ind_{sigla}_rbd']:
                if patron in df.columns:
                    renombres[patron] = f'idps_{sigla}_{nivel}'
        df = df.rename(columns=renombres)

        cols_finales = cols_meta + [f'idps_{s}_{nivel}' for s in ['am', 'cc', 'pf', 'hv'] if f'idps_{s}_{nivel}' in df.columns]
        df = df[cols_finales]

    # --- Normalización común: llave RBD blindada + downcasting ---
    df['rbd'] = pd.to_numeric(df['rbd'], errors='coerce').astype('Int64')
    if df['rbd'].isnull().any():
        print(f"AVISO   | {clave:<14} | {int(df['rbd'].isnull().sum())} filas sin RBD válido eliminadas")
        df = df.dropna(subset=['rbd'])
    df['rbd'] = df['rbd'].astype(str)

    cols_idps = [c for c in df.columns if c.startswith('idps_')]
    for col in cols_idps:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

    if 'cod_depe2' in df.columns:
        df['cod_depe2'] = pd.to_numeric(df['cod_depe2'], errors='coerce').astype('Int64').astype('string')
    if 'cod_grupo' in df.columns:
        df['cod_grupo'] = pd.to_numeric(df['cod_grupo'], errors='coerce').astype('Int64').astype('string')
    if 'cod_rural_rbd' in df.columns:
        df['ES_RURAL'] = (pd.to_numeric(df['cod_rural_rbd'], errors='coerce') == 2)
        df = df.drop(columns=['cod_rural_rbd'])

    dfs_idps_limpios[clave] = df
    print(f"OK      | {clave:<14} | {df.shape[0]:>6} filas | {df.columns.tolist()}")

print(f"\nTotal de DataFrames limpios: {len(dfs_idps_limpios)} de {len(dfs_idps_brutos)}")

# ==============================================================================
# AUDITORÍA
# ==============================================================================
print("\n--- Auditoría de tipos y RBD duplicados ---")
for clave, df in dfs_idps_limpios.items():
    dup = int(df['rbd'].duplicated().sum())
    print(f"{clave:<14} | filas: {df.shape[0]:>6} | RBD duplicados: {dup} | columnas idps: {[c for c in df.columns if c.startswith('idps_')]}")

OK      | idps_4b_2016   |   7390 filas | ['rbd', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b']
OK      | idps_2m_2016   |   2891 filas | ['rbd', 'idps_am_2m', 'idps_cc_2m', 'idps_pf_2m', 'idps_hv_2m']
AVISO   | idps_4b_2017   | 1 filas sin RBD válido eliminadas
OK      | idps_4b_2017   |   7383 filas | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b', 'ES_RURAL']
OK      | idps_8b_2017   |   5982 filas | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_8b', 'idps_cc_8b', 'idps_pf_8b', 'idps_hv_8b', 'ES_RURAL']
OK      | idps_2m_2017   |   2912 filas | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_2m', 'idps_cc_2m', 'idps_pf_2m', 'idps_hv_2m', 'ES_RURAL']
OK      | idps_4b_2018   |   7414 filas | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b', 'ES_RURAL']
OK      | idps_6b_2018   |   7322 filas | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_6b', 'idps_

Un patrón numérico salta a la vista y vale la pena señalarlo antes de seguir: hay una caída de aproximadamente el 11% en el número de colegios entre 2019 y 2022 (5.960→2018 8ºB pasa a categorías más chicas: 7.414 en 4ºB 2018 → 6.591 en 4ºB 2022, una caída de ~823 colegios, ~11%). Es mayor que la caída natural que veníamos observando en SIMCE por cierre de escuelas rurales — probablemente refleja que el IDPS depende de una tasa de respuesta mínima al cuestionario para publicar el dato del colegio (a diferencia de SIMCE, que es una prueba obligatoria), así que colegios con baja participación en la encuesta simplemente no aparecen en el archivo. Es una hipótesis razonable, no una certeza — si tu glosa de 2022 o el documento técnico del IDPS menciona un umbral mínimo de respuestas, seria bueno confirmarlo y documentarlo como otra fuente de "ausencia estructural", análoga a los NaN de educación especial que ya documentamos en SNED.

In [10]:
# ==============================================================================
# CELDA 4: CONSOLIDACIÓN DE BIENIOS IDPS (concat vertical + groupby.mean)
# ==============================================================================
bienios_idps = {
    '2016-17': ['idps_4b_2016', 'idps_2m_2016', 'idps_4b_2017', 'idps_8b_2017', 'idps_2m_2017'],
    '2018-19': ['idps_4b_2018', 'idps_6b_2018', 'idps_2m_2018', 'idps_8b_2019'],
    '2022-23': ['idps_4b_2022', 'idps_2m_2022', 'idps_4b_2023', 'idps_2m_2023'],
    '2024-25': ['idps_4b_2024', 'idps_6b_2024', 'idps_2m_2024', 'idps_4b_2025', 'idps_8b_2025', 'idps_2m_2025'],
}

dfs_idps_bienios = {}

for bienio, claves in bienios_idps.items():
    faltantes = [c for c in claves if c not in dfs_idps_limpios]
    if faltantes:
        print(f"ALERTA | Bienio {bienio}: fuentes faltantes {faltantes}. Se omite.")
        continue

    pool = pd.concat([dfs_idps_limpios[c] for c in claves], ignore_index=True)
    df_bienio = pool.groupby('rbd', as_index=False).mean(numeric_only=True)

    cols_atributos = [c for c in ['nom_rbd', 'cod_depe2', 'cod_grupo'] if c in pool.columns]
    atributos = pool.groupby('rbd', as_index=False)[cols_atributos].first()
    df_bienio = pd.merge(atributos, df_bienio, on='rbd', how='right')

    # ES_RURAL (booleano) requiere trato aparte: promedio de True/False no tiene sentido,
    # tomamos el valor más frecuente (moda) por colegio
    if 'ES_RURAL' in pool.columns:
        moda_rural = pool.groupby('rbd')['ES_RURAL'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        df_bienio = pd.merge(df_bienio, moda_rural.reset_index(), on='rbd', how='left')

    cols_metricas = df_bienio.select_dtypes(include=['float64', 'float32']).columns
    df_bienio[cols_metricas] = df_bienio[cols_metricas].astype('float32')
    df_bienio['BIENIO'] = bienio

    dfs_idps_bienios[bienio] = df_bienio

    dup = int(df_bienio['rbd'].duplicated().sum())
    print(f"Bienio {bienio} | {df_bienio.shape[0]:>5} colegios | RBD duplicados: {dup} | {df_bienio.columns.tolist()}")

for bienio, df in dfs_idps_bienios.items():
    print(f"\n### df.info() — Bienio IDPS {bienio} ###")
    print(df.info())

Bienio 2016-17 |  8610 colegios | RBD duplicados: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b', 'idps_am_2m', 'idps_cc_2m', 'idps_pf_2m', 'idps_hv_2m', 'idps_am_8b', 'idps_cc_8b', 'idps_pf_8b', 'idps_hv_8b', 'ES_RURAL', 'BIENIO']
Bienio 2018-19 |  8534 colegios | RBD duplicados: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b', 'ES_RURAL_x', 'idps_am_6b', 'idps_cc_6b', 'idps_pf_6b', 'idps_hv_6b', 'idps_am_2m', 'idps_cc_2m', 'idps_pf_2m', 'idps_hv_2m', 'idps_am_8b', 'idps_cc_8b', 'idps_pf_8b', 'idps_hv_8b', 'ES_RURAL_y', 'BIENIO']
Bienio 2022-23 |  7711 colegios | RBD duplicados: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'idps_am_4b', 'idps_cc_4b', 'idps_hv_4b', 'idps_pf_4b', 'ES_RURAL_x', 'idps_am_2m', 'idps_cc_2m', 'idps_hv_2m', 'idps_pf_2m', 'ES_RURAL_y', 'BIENIO']
Bienio 2024-25 |  7863 colegios | RBD duplicados: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'i

In [11]:
# ==============================================================================
# CELDA 4: CONSOLIDACIÓN DE BIENIOS IDPS (corregida)
# ==============================================================================
bienios_idps = {
    '2016-17': ['idps_4b_2016', 'idps_2m_2016', 'idps_4b_2017', 'idps_8b_2017', 'idps_2m_2017'],
    '2018-19': ['idps_4b_2018', 'idps_6b_2018', 'idps_2m_2018', 'idps_8b_2019'],
    '2022-23': ['idps_4b_2022', 'idps_2m_2022', 'idps_4b_2023', 'idps_2m_2023'],
    '2024-25': ['idps_4b_2024', 'idps_6b_2024', 'idps_2m_2024', 'idps_4b_2025', 'idps_8b_2025', 'idps_2m_2025'],
}

def primer_no_nulo(s):
    s_validos = s.dropna()
    return s_validos.iloc[0] if len(s_validos) > 0 else pd.NA

dfs_idps_bienios = {}

for bienio, claves in bienios_idps.items():
    faltantes = [c for c in claves if c not in dfs_idps_limpios]
    if faltantes:
        print(f"ALERTA | {bienio}: faltan {faltantes}")
        continue

    pool = pd.concat([dfs_idps_limpios[c] for c in claves], ignore_index=True)

    cols_metricas = [c for c in pool.columns if c.startswith('idps_')]
    df_bienio = pool.groupby('rbd', as_index=False)[cols_metricas].mean()

    cols_atrib = [c for c in ['nom_rbd', 'cod_depe2', 'cod_grupo', 'ES_RURAL'] if c in pool.columns]
    atributos = pool.groupby('rbd', as_index=False)[cols_atrib].agg(primer_no_nulo)

    df_bienio = pd.merge(atributos, df_bienio, on='rbd', how='right')
    df_bienio[cols_metricas] = df_bienio[cols_metricas].astype('float32')
    df_bienio['BIENIO'] = bienio

    dfs_idps_bienios[bienio] = df_bienio
    dup = int(df_bienio['rbd'].duplicated().sum())
    print(f"Bienio {bienio} | {df_bienio.shape[0]:>5} colegios | dup: {dup} | {df_bienio.columns.tolist()}")

Bienio 2016-17 |  8610 colegios | dup: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'ES_RURAL', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b', 'idps_am_2m', 'idps_cc_2m', 'idps_pf_2m', 'idps_hv_2m', 'idps_am_8b', 'idps_cc_8b', 'idps_pf_8b', 'idps_hv_8b', 'BIENIO']
Bienio 2018-19 |  8534 colegios | dup: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'ES_RURAL', 'idps_am_4b', 'idps_cc_4b', 'idps_pf_4b', 'idps_hv_4b', 'idps_am_6b', 'idps_cc_6b', 'idps_pf_6b', 'idps_hv_6b', 'idps_am_2m', 'idps_cc_2m', 'idps_pf_2m', 'idps_hv_2m', 'idps_am_8b', 'idps_cc_8b', 'idps_pf_8b', 'idps_hv_8b', 'BIENIO']
Bienio 2022-23 |  7711 colegios | dup: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'ES_RURAL', 'idps_am_4b', 'idps_cc_4b', 'idps_hv_4b', 'idps_pf_4b', 'idps_am_2m', 'idps_cc_2m', 'idps_hv_2m', 'idps_pf_2m', 'BIENIO']
Bienio 2024-25 |  7863 colegios | dup: 0 | ['rbd', 'nom_rbd', 'cod_depe2', 'cod_grupo', 'ES_RURAL', 'idps_am_4b', 'idps_cc_4b', 'idps_hv_4b', 'idps_pf_4b', 'idps_am_6

In [12]:
# ==============================================================================
# CELDA 5: CHECKPOINT — MATRIZ MAESTRA IDPS
# ==============================================================================
df_idps_maestro = pd.concat(list(dfs_idps_bienios.values()), ignore_index=True)

dup = int(df_idps_maestro.duplicated(subset=['rbd', 'BIENIO']).sum())
print(f"Duplicados (rbd, BIENIO): {dup}")
print(df_idps_maestro['BIENIO'].value_counts())

RUTA_SALIDA = RUTA_PROCESADOS
df_idps_maestro.to_parquet(RUTA_SALIDA + 'idps_maestro_bienios.parquet', index=False)
print(f"\nGuardado en: {RUTA_SALIDA}idps_maestro_bienios.parquet")
print(df_idps_maestro.shape)

Duplicados (rbd, BIENIO): 0
BIENIO
2016-17    8610
2018-19    8534
2024-25    7863
2022-23    7711
Name: count, dtype: int64

Guardado en: /content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/idps_maestro_bienios.parquet
(32718, 22)


In [13]:
# ==============================================================================
# CELDA 6: INTEGRAR IDPS 2018-19 A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
df_modelo = pd.read_parquet(RUTA_SALIDA + 'tabla_modelo_final.parquet')

idps_1819 = df_idps_maestro[df_idps_maestro['BIENIO'] == '2018-19'].copy()
cols_idps = [c for c in idps_1819.columns if c.startswith('idps_')]
idps_1819 = idps_1819[['rbd'] + cols_idps]

df_modelo_v2 = pd.merge(df_modelo, idps_1819, on='rbd', how='left', validate='one_to_one')

print(f"Filas antes: {len(df_modelo)} | después: {len(df_modelo_v2)}")
print(f"Colegios con IDPS: {df_modelo_v2[cols_idps[0]].notna().sum()} / {len(df_modelo_v2)}")

df_modelo_v2.to_parquet(RUTA_SALIDA + 'tabla_modelo_final_v2.parquet', index=False)
print(df_modelo_v2.shape)

Filas antes: 7754 | después: 7754
Colegios con IDPS: 6587 / 7754
(7754, 40)
